In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

from azure.core.exceptions import ResourceExistsError, ResourceNotFoundError
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

# 이 노트북(azure-doc-ai-service/notebooks/)의 부모 폴더(azure-doc-ai-service/)에 있는 .env를 로드
ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

CONNECTION_STRING = os.environ["AZURE_STORAGE_CONNECTION_STRING"]
CONTAINER_NAME = os.environ.get("AZURE_STORAGE_CONTAINER_NAME", "documents")

# CRUD 테스트에 사용할 blob(파일) 이름
BLOB_NAME = "test/hello.txt"

print("CONTAINER_NAME:", CONTAINER_NAME)
print("BLOB_NAME:", BLOB_NAME)

In [ ]:
blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)

print("연결 성공, 계정 내 컨테이너 목록(최대 10개):")
for i, container in enumerate(blob_service_client.list_containers()):
    if i >= 10:
        print("  ...")
        break
    print(f"  - {container.name}")

In [ ]:
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

try:
    container_client.create_container()
    print(f"컨테이너 '{CONTAINER_NAME}' 생성됨")
except ResourceExistsError:
    print(f"컨테이너 '{CONTAINER_NAME}' 이미 존재함 (그대로 사용)")


def create_blob(blob_name: str, content: str) -> None:
    """blob_name이 이미 있으면 에러를 내도록(overwrite=False) 만들어서 진짜 '생성'만 테스트한다."""
    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(content.encode("utf-8"), overwrite=False)
    print(f"'{blob_name}' 생성 완료 ({len(content)}자)")


initial_content = f"hello from blob_storage_test.ipynb (created_at={datetime.now(timezone.utc).isoformat()})"

try:
    create_blob(BLOB_NAME, initial_content)
except ResourceExistsError:
    print(f"'{BLOB_NAME}'가 이미 존재합니다. 삭제 후 다시 실행하거나 BLOB_NAME을 바꾸세요.")

In [ ]:
def list_blobs(name_starts_with: str | None = None) -> list[str]:
    return [b.name for b in container_client.list_blobs(name_starts_with=name_starts_with)]


def read_blob(blob_name: str) -> str:
    blob_client = container_client.get_blob_client(blob_name)
    data = blob_client.download_blob().readall()
    return data.decode("utf-8")


print(f"컨테이너 '{CONTAINER_NAME}' 안의 blob 목록:")
for name in list_blobs():
    print(f"  - {name}")

print(f"\n'{BLOB_NAME}' 내용:")
print(read_blob(BLOB_NAME))

In [ ]:
def update_blob(blob_name: str, content: str) -> None:
    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(content.encode("utf-8"), overwrite=True)
    print(f"'{blob_name}' 갱신 완료 ({len(content)}자)")


updated_content = f"updated content (updated_at={datetime.now(timezone.utc).isoformat()})"
update_blob(BLOB_NAME, updated_content)

print("\n갱신 후 내용 재조회:")
print(read_blob(BLOB_NAME))

In [ ]:
def delete_blob(blob_name: str) -> None:
    blob_client = container_client.get_blob_client(blob_name)
    try:
        blob_client.delete_blob()
        print(f"'{blob_name}' 삭제 완료")
    except ResourceNotFoundError:
        print(f"'{blob_name}'가 이미 없습니다.")


delete_blob(BLOB_NAME)

print("\n삭제 후 blob 목록:")
for name in list_blobs():
    print(f"  - {name}")